# Figure S3 - NTC comparison of iTF vs iMG

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc 
import scanpy.external as sce
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap

sys.path.append('../utils')

import anndata as ad 
import fast_pyucell as pyucell
import signature_heatmaps as signature_heatmaps
import factor_labels as factor_labels
import muon as mu


## Load Data

In [ ]:
data_dir = "<path to processed data>"

cite_6tf_path = os.path.join(data_dir, "cite_6tf_cleaned_revisions.h5mu")
cite_imgl_path = os.path.join(data_dir, "cite_imgl_cleaned_revisions.h5mu")
merged_6tf_path = os.path.join(data_dir, "adata_revisions_merged_6tf.h5ad")

In [ ]:
mdata_dict = {}
mdata_dict['cite_6tf'] = mu.read_h5mu(cite_6tf_path)
mdata_dict['cite_imgl'] = mu.read_h5mu(cite_imgl_path)

adata_dict = {}
adata_dict['merged_6tf'] = sc.read_h5ad(merged_6tf_path)
adata_dict['cite_6tf'] = mdata_dict['cite_6tf'].mod['rna'].copy()
adata_dict['cite_imgl'] = mdata_dict['cite_imgl'].mod['rna'].copy()

In [ ]:
signature_cols_ordered = ['homeostatic_score_ucell',
 'interferon_score_ucell',
 'chemokine_score_ucell',
 'antigen_presenting_score_ucell',
 'dam_score_ucell',
 'lipid_dam_score_ucell']

## Masking for analysis
to exclude ntc_g5 + foxk1_g2 + mixscale_cutoff >= 0

In [ ]:
guides_to_exclude = ['FOXK1_g2', 'non-targeting_g5']

adata_6tf_clean = adata_dict['merged_6tf'][~adata_dict['merged_6tf'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
adata_imgl_clean = adata_dict['cite_imgl'][~adata_dict['cite_imgl'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
adata_6tf_clean.shape, adata_imgl_clean.shape

In [ ]:
print(mdata_dict['cite_6tf'].shape, mdata_dict['cite_imgl'].shape)
mdata_6tf_clean = mdata_dict['cite_6tf'][~mdata_dict['cite_6tf'].mod['rna'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
mdata_imgl_clean = mdata_dict['cite_imgl'][~mdata_dict['cite_imgl'].mod['rna'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
mdata_6tf_clean.shape, mdata_imgl_clean.shape

In [ ]:
mixscale_col = "mixscale_score"

In [ ]:
adata_6tf_masked = adata_6tf_clean[adata_6tf_clean.obs[mixscale_col] >= 0].copy()
adata_imgl_masked = adata_imgl_clean[adata_imgl_clean.obs[mixscale_col] >= 0].copy()
adata_6tf_masked.shape, adata_imgl_masked.shape

In [ ]:
adata_masked_dict = {}
adata_masked_dict['iTF-MG'] = adata_6tf_masked
adata_masked_dict['iMG'] = adata_imgl_masked

In [ ]:
genes_to_include = ['DNMT1', 'IRF9', 'STAT2', 'SMAD3', 'PRDM1', 'ZNF532']
itf_genes = ['DNMT1', 'IRF9', 'STAT2', 'SMAD3']
img_genes = ['PRDM1', 'ZNF532']

# Getting NTCs

In [ ]:
adata_imgl_clean_ntc = adata_imgl_clean[adata_imgl_clean.obs['perturbed_gene'] == 'NTC'].copy()
adata_6tf_clean_ntc = adata_6tf_clean[adata_6tf_clean.obs['perturbed_gene'] == 'NTC'].copy()

In [ ]:
adata_imgl_clean_ntc.obs['model'] = "iMG"
adata_6tf_clean_ntc.obs['model'] = "iTF-MG"

# Table 15: Comparing NTC Expression of Macrophage and Microglia Genes

In [ ]:
macrophage_genes = pd.read_csv("../ref/AbudMicrogliaMacrophageList.txt",
                               header=None)
genes_to_test = macrophage_genes[0].tolist()

In [ ]:
genes_in_6tf = set(adata_6tf_clean_ntc.var[adata_6tf_clean_ntc.var.index.isin(genes_to_test)].index.tolist())
genes_in_imgl = set(adata_imgl_clean_ntc.var[adata_imgl_clean_ntc.var.index.isin(genes_to_test)].index.tolist())

In [ ]:
common_genes = list(genes_in_6tf.intersection(genes_in_imgl))
len(common_genes)

In [ ]:
adata_imgl_clean_ntc.obs['barcode'] = adata_imgl_clean_ntc.obs.index.tolist()
adata_6tf_clean_ntc.obs['barcode'] = adata_6tf_clean_ntc.obs.index.tolist()

In [ ]:
adata_combined = ad.concat([adata_imgl_clean_ntc, adata_6tf_clean_ntc], join="outer")

In [ ]:
adata_combined_mac = adata_combined[:, common_genes]

In [ ]:
sc.set_figure_params(dpi=300, dpi_save=300)
mp = sc.pl.matrixplot(adata_combined_mac, var_names=common_genes, 
                 groupby="model",
                 cmap='viridis', swap_axes=False,
                 use_raw=False,
                 figsize=(40,3),
                 save="fig-se_unscaled_ntc_comparison_macrophage.svg",
                 vmax=1, 
                 return_fig=True)

In [ ]:
df = mp.values_df
df.T.to_csv("macrophage_ntc_comparison_values.csv")

# Figure S3: NTC Integration of Models

In [ ]:
adata_combined = ad.concat([adata_imgl_clean_ntc, adata_6tf_clean_ntc], join="outer")

In [ ]:
del adata_combined.obsm['X_pca_harmony']
del adata_combined.obsm['X_pca']
del adata_combined.obsm['X_umap']

In [ ]:
sc.pp.highly_variable_genes(adata_combined)

In [ ]:
sc.tl.pca(adata_combined, random_state=42)

In [ ]:
sce.pp.harmony_integrate(adata_combined, key='model', random_state=42, max_iter_harmony=30, theta=4, sigma=0.3)


In [ ]:
sc.pp.neighbors(adata_combined, use_rep="X_pca_harmony", random_state=42, key_added="neighbors_harmony")


In [ ]:
sc.tl.umap(adata_combined, random_state=42, neighbors_key="neighbors_harmony")
sc.tl.leiden(adata_combined, resolution=1.0, random_state=42, key_added="leiden_1.0", neighbors_key="neighbors_harmony")
sc.tl.leiden(adata_combined, resolution=0.8, random_state=42, key_added="leiden_0.8", neighbors_key="neighbors_harmony")
sc.tl.leiden(adata_combined, resolution=0.5, random_state=42, key_added="leiden_0.5", neighbors_key="neighbors_harmony")

In [ ]:
model_palette = {"iTF-MG":"#c0baad", "iMG": "#462D21"}

In [ ]:
sns.set_theme('poster', style="white")
fig = sc.pl.umap(adata_combined, color=['model'],
                 palette=model_palette,
                 size=50,
                 frameon=False,
                 title="",
                 show=False,
                 return_fig=True)
fig.set_dpi(300)
fig.savefig("figures/ntc_integrated_umap.svg", bbox_inches="tight")
fig.show()

In [ ]:
fig_dir = "figures/"

In [ ]:
sns.set_theme("poster", "white")
for signature in signature_cols_ordered:
    fig = plt.figure(figsize=(5,5), dpi=300)
    sns.kdeplot(adata_combined.obs,
                    x=signature,
                    hue="model",
                    legend=False,
                    palette=model_palette,
                    fill=False,
                    linewidth = 6,
                    cumulative=False,
                    common_norm=False
                    )
    fig.savefig(f"{fig_dir}/ntc_comparison_{signature}_histogram_norm.svg")


# Violin Plots for NTC

In [ ]:
adata_combined = ad.concat([adata_imgl_clean_ntc, adata_6tf_clean_ntc], join="outer")

In [ ]:
state_genes_df = pd.read_csv("<path to reference state gene families, table S2 tab 2>")
state_genes_df

In [ ]:
states_dict = {}
for name, adata, in adata_dict.items():
    print("EXP: ", name)
    states_dict[name] = pyucell.get_state_genes(adata, reference=state_genes_df, suffix='')

In [ ]:
signature_cols_ordered_colors = {
    "homeostatic_score_ucell": "#107B35",
    "interferon_score_ucell": "#5FADAF",
    "chemokine_score_ucell": "#A335C2",
    "antigen_presenting_score_ucell": "#FDAC10",
    "dam_score_ucell": "#BF0063",
    "lipid_dam_score_ucell": "#E76333",
}

In [ ]:
state_to_color_key = {
    "Interferon_score":        "interferon_score_ucell",
    "Disease-responsive_score": "dam_score_ucell",
    "Chemokine_score":         "chemokine_score_ucell",
    "Homeostatic_score":       "homeostatic_score_ucell",
    "Antigen-presenting_score": "antigen_presenting_score_ucell",
    "Lipid-high_score":        "lipid_dam_score_ucell",
}

all_states = list(dict.fromkeys(
    list(states_dict.get("merged_6tf", {}).keys()) +
    list(states_dict.get("cite_imgl", {}).keys())
))

In [ ]:
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams["font.style"] = "italic"

def make_color_cmap(hex_color, name="custom"):
    return LinearSegmentedColormap.from_list(name, ["black", hex_color])

In [ ]:
for state in all_states:
    if len(states_dict["merged_6tf"][state]) > len(states_dict["cite_imgl"][state]):
        model_with_max_genes = "merged_6tf"
    else:
        model_with_max_genes = "cite_imgl"
        
    genes = states_dict[model_with_max_genes][state]
    
    color = signature_cols_ordered_colors[state_to_color_key[state]]
    cmap = make_color_cmap(color)

    sc.pl.stacked_violin(
        adata_combined,
        var_names=genes,
        groupby="model",
        swap_axes=True,
        show=False,
        cmap=cmap,
        vmax=2
    )
    fig = plt.gcf()
    fig.suptitle(state, fontsize=14, fontweight="bold")
    fig.savefig(f"stacked_violin_ntc_{state}.svg", bbox_inches="tight", dpi=300)
    plt.close(fig)